# Лабораторная работа № 6

In [14]:
import pandas as pd
import numpy as np

import re
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/ivan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
df = pd.read_csv("../../data/scientific_articles.csv")
print(df.head())
print(df.info())

print(df.isnull().sum())
df = df.dropna()

                                                 url  \
0  https://link.springer.com/article/10.1007/s007...   
1  https://link.springer.com/article/10.1007/s007...   
2  https://link.springer.com/article/10.1007/s007...   
3  https://link.springer.com/article/10.1007/s007...   
4  https://link.springer.com/article/10.1007/s007...   

                                               title  \
0  Novel carbon material with potential applicati...   
1  Specific conductivities of tetraalkylammonium ...   
2  Development of electrochemical sensor for quan...   
3  Synthesis of thioamides from Schiff bases and ...   
4  Mechanistic study and computational analysis o...   

                                            abstract  
0  AbstractLead-acid batteries (LABs) are one of ...  
1  AbstractSearching for environmentally friendly...  
2  AbstractThis study presents the development of...  
3  AbstractThioamides play an important role in p...  
4  AbstractThe mechanism of electrochemical oxida..

In [16]:
df['text'] = df['title'] + " " + df['abstract']


def assign_label(text):
    text = text.lower()

    if "battery" in text or "energy" in text:
        return "energy"
    elif "chemical" in text or "reaction" in text:
        return "chemistry"
    elif "material" in text or "carbon" in text:
        return "materials"
    elif "model" in text or "algorithm" in text:
        return "ai"
    else:
        return "other"


df['label'] = df['text'].apply(assign_label)
print(df['label'].value_counts())

label
other        69486
ai           13222
chemistry     8367
energy        4491
materials     4433
Name: count, dtype: int64


In [17]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return text

In [19]:
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english'))


def normalize(text):
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)

In [20]:
def pipeline(text):
    text = preprocess(text)
    text = normalize(text)
    return text


X_train_proc = X_train.apply(pipeline)
X_test_proc = X_test.apply(pipeline)


In [21]:
bow = CountVectorizer(ngram_range=(1, 1))

X_train_bow = bow.fit_transform(X_train_proc)
X_test_bow = bow.transform(X_test_proc)

In [22]:
tfidf = TfidfVectorizer(ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train_proc)
X_test_tfidf = tfidf.transform(X_test_proc)

In [23]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

preds = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, preds))
print("F1:", f1_score(y_test, preds, average="macro"))

Accuracy: 0.88905
F1: 0.7493089382981822


In [24]:
# BoW
model.fit(X_train_bow, y_train)
preds_bow = model.predict(X_test_bow)

print("BoW Accuracy:", accuracy_score(y_test, preds_bow))

# TF-IDF
model.fit(X_train_tfidf, y_train)
preds_tfidf = model.predict(X_test_tfidf)

print("TF-IDF Accuracy:", accuracy_score(y_test, preds_tfidf))

BoW Accuracy: 0.95175
TF-IDF Accuracy: 0.88905
